# Grouped Shared-Prefix Training Prototype

In [ ]:
import importlib
import random
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

sys.path.insert(0, "kev")

import data.persona as _persona_mod
import data.base as _base_mod
import data.issp as _issp_mod
import data.twin2k as _twin2k_mod
import data.grummon as _grummon_mod
import models.kev_pointer as _kev_pointer_mod
import training.plotter as _plotter_mod
import training.trainer as _trainer_mod
import training as _training_pkg

for _mod in (
    _persona_mod,
    _base_mod,
    _issp_mod,
    _twin2k_mod,
    _grummon_mod,
    _kev_pointer_mod,
    _plotter_mod,
    _trainer_mod,
    _training_pkg,
):
    importlib.reload(_mod)

from data.base import (
    build_persona_dataset,
    load_split_qids,
)
from data.grummon import load_grummon
from data.issp import load_issp_df, load_issp_eval
from data.twin2k import load_twin2k_df
from training import GroupedTrainer, free_gpu

## 1. Load & Build Persona Datasets

In [ ]:
INCLUDE_COUNTRY = False
TEST10_SEED = 42

issp_qids = load_split_qids("splits/split.json")
issp_train_df = load_issp_df(qids=issp_qids["train"])
twin_df = load_twin2k_df()
train_df = pd.concat([issp_train_df, twin_df], ignore_index=True)

train_records_issp_twin2k_resp = build_persona_dataset(
    train_df, attr_sets=None, include_country=INCLUDE_COUNTRY, desc="train_records_resp"
)

issp_test10_qids = random.Random(TEST10_SEED).sample(sorted(issp_qids["test"]), 10)
issp_resp_valid_grouped = load_issp_eval(
    qids=issp_qids["valid"], attr_sets=None, include_country=INCLUDE_COUNTRY, grouped=True, name="valid", valid=False
)
issp_resp_test_grouped = load_issp_eval(
    qids=issp_qids["test"], attr_sets=None, include_country=INCLUDE_COUNTRY, grouped=True, name="test", valid=False
)
issp_resp_test10_grouped = load_issp_eval(
    qids=issp_test10_qids, attr_sets=None, include_country=INCLUDE_COUNTRY, grouped=True, name="test_10", valid=True
)
grummon_grouped = load_grummon(
    attr_sets=None, include_country=INCLUDE_COUNTRY, grouped=True, valid=False
)

eval_records = [issp_resp_test10_grouped, grummon_grouped]

print(
    f"Train records : {len(train_records_issp_twin2k_resp):,}\n"
    f"Eval sets     : "
    f"valid={len(issp_resp_valid_grouped['records']):,} ({len(issp_resp_valid_grouped['packed_views']):,} states), "
    f"test={len(issp_resp_test_grouped['records']):,} ({len(issp_resp_test_grouped['packed_views']):,} states), "
    f"test_10={len(issp_resp_test10_grouped['records']):,} ({len(issp_resp_test10_grouped['packed_views']):,} states), "
    f"grummon={len(grummon_grouped['records']):,} ({len(grummon_grouped['packed_views']):,} states)"
)

## 2. Configure & Instantiate `GroupedTrainer`


In [ ]:
# Choose model preset ("9b" or "0.8b"):
PRESET = "9b"

if PRESET == "9b":
    trainer_cfg = dict(
        base_model="jaredpalmer/kev-9b",
        save_dir="runs/kev-9b-resp-issp+twin2k-grouped",
        weights_dtype="bf16",
        use_bf16=True,
        gradient_checkpointing=True,
        max_questions_per_group=128,
        max_chars_per_group=48_000,
        eval_batch_size=64,
    )
elif PRESET == "0.8b":
    trainer_cfg = dict(
        base_model="jaredpalmer/kev-0.8b",
        save_dir="runs/kev-0.8b-resp-issp+twin2k-grouped",
        weights_dtype="fp32",
        use_bf16=False,
        gradient_checkpointing=False,
        max_questions_per_group=64,
        max_chars_per_group=20_000,
        eval_batch_size=128,
    )
else:
    raise ValueError(f"Unknown PRESET: {PRESET!r}")

trainer = None
free_gpu()

trainer = GroupedTrainer(
    train_records=train_records_issp_twin2k_resp,
    eval_records=eval_records,
    lora_rank=64,
    epochs=5,
    lr=5e-5,
    batch_size=1,
    grad_accum_steps=12,
    warmup_steps=2000,
    val_every=300,
    samples_per_epoch=None,
    resume=False,
    **trainer_cfg,
)

display(pd.Series(trainer.describe(), name="trainer_config").to_frame())

## 3. Optional: Probe VRAM directly on `trainer`

`trainer.probe_vram(grid)` reuses `trainer.train_one_step` with a zero-LR AdamW optimizer so it measures exact steady-state VRAM without mutating model weights or creating empty plots.

In [ ]:
df_probe = trainer.probe_vram([
    (64, 24_000),
    (96, 36_000),
    (128, 48_000),
    (160, 60_000),
])
display(df_probe)

# You can also update grouping on the fly and recompute the schedule:
# trainer.set_grouping(max_questions_per_group=160, max_chars_per_group=60_000)

## 4. Train

`trainer.train()` initializes the composed `TrainingPlotter` display, runs the grouped shared-prefix training loop, saves `best`/`last` checkpoints and lightweight `trainer_state.pt` (`epoch_rng_state`), and automatically frees optimizer/scheduler/gradient buffers in `finally:`.

In [ ]:
plotter = trainer.train()